# Docker & Containerization Assignment

**Task:** Install Docker and create Dockerfiles to containerize applications.

This notebook shows Dockerfiles for three common application types (Node.js, Python/Flask,
and Java/Spring Boot-style) so you can pick whichever matches your stack, plus best-practice
notes (multi-stage builds, `.dockerignore`, non-root user).

## 1. Install Docker
```bash
# Ubuntu/Debian
curl -fsSL https://get.docker.com -o get-docker.sh
sudo sh get-docker.sh
sudo usermod -aG docker $USER   # allow running docker without sudo (re-login required)
docker --version
docker run hello-world           # sanity check
```
On Windows/Mac, install **Docker Desktop** instead and verify with the same `docker --version` / `docker run hello-world` commands.

## 2. Dockerfile — Node.js app

In [1]:
import os
os.makedirs('project/node-app', exist_ok=True)

with open('project/node-app/index.js', 'w') as f:
    f.write('''const http = require('http');
http.createServer((req, res) => {
  res.writeHead(200, {'Content-Type': 'text/plain'});
  res.end('Hello from the containerized Node app!\n');
}).listen(3000, () => console.log('listening on 3000'));
''')

with open('project/node-app/package.json', 'w') as f:
    f.write('{ "name": "node-app", "version": "1.0.0", "main": "index.js" }')

with open('project/node-app/Dockerfile', 'w') as f:
    f.write('''# ---- Build stage ----
FROM node:18-alpine AS build
WORKDIR /app
COPY package*.json ./
RUN npm install --production

# ---- Runtime stage ----
FROM node:18-alpine
WORKDIR /app
COPY --from=build /app/node_modules ./node_modules
COPY . .
RUN addgroup -S appgroup && adduser -S appuser -G appgroup
USER appuser
EXPOSE 3000
CMD ["node", "index.js"]
''')

with open('project/node-app/.dockerignore', 'w') as f:
    f.write("node_modules\n.git\n*.log\n")

print("Created project/node-app/{index.js,package.json,Dockerfile,.dockerignore}")
print("Build: docker build -t node-app project/node-app")
print("Run:   docker run -p 3000:3000 node-app")


Created project/node-app/{index.js,package.json,Dockerfile,.dockerignore}
Build: docker build -t node-app project/node-app
Run:   docker run -p 3000:3000 node-app


## 3. Dockerfile — Python / Flask app

In [2]:
import os
os.makedirs('project/python-app', exist_ok=True)

with open('project/python-app/app.py', 'w') as f:
    f.write('''from flask import Flask
app = Flask(__name__)

@app.route("/")
def hello():
    return "Hello from the containerized Flask app!"

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000)
''')

with open('project/python-app/requirements.txt', 'w') as f:
    f.write("flask\n")

with open('project/python-app/Dockerfile', 'w') as f:
    f.write('''FROM python:3.11-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY . .
RUN useradd -m appuser
USER appuser
EXPOSE 5000
CMD ["python", "app.py"]
''')
print("Created project/python-app/{app.py,requirements.txt,Dockerfile}")
print("Build: docker build -t python-app project/python-app")
print("Run:   docker run -p 5000:5000 python-app")


Created project/python-app/{app.py,requirements.txt,Dockerfile}
Build: docker build -t python-app project/python-app
Run:   docker run -p 5000:5000 python-app


## 4. Dockerfile — Java (Spring Boot style, Maven multi-stage build)

In [3]:
import os
os.makedirs('project/java-app/src/main/java/com/example', exist_ok=True)

with open('project/java-app/pom.xml', 'w') as f:
    f.write('''<project xmlns="http://maven.apache.org/POM/4.0.0">
  <modelVersion>4.0.0</modelVersion>
  <groupId>com.example</groupId>
  <artifactId>java-app</artifactId>
  <version>1.0.0</version>
  <properties>
    <maven.compiler.source>17</maven.compiler.source>
    <maven.compiler.target>17</maven.compiler.target>
  </properties>
  <build>
    <plugins>
      <plugin>
        <groupId>org.apache.maven.plugins</groupId>
        <artifactId>maven-jar-plugin</artifactId>
        <version>3.2.0</version>
        <configuration>
          <archive>
            <manifest>
              <mainClass>com.example.Main</mainClass>
            </manifest>
          </archive>
        </configuration>
      </plugin>
    </plugins>
  </build>
</project>
''')

with open('project/java-app/src/main/java/com/example/Main.java', 'w') as f:
    f.write('''package com.example;

public class Main {
    public static void main(String[] args) {
        System.out.println("Hello from containerized Java app!");
    }
}
''')

with open('project/java-app/Dockerfile', 'w') as f:
    f.write('''# ---- Build stage ----
FROM maven:3.9-eclipse-temurin-17 AS build
WORKDIR /app
COPY pom.xml .
RUN mvn dependency:go-offline
COPY src ./src
RUN mvn clean package -DskipTests

# ---- Runtime stage ----
FROM eclipse-temurin:17-jre-alpine
WORKDIR /app
COPY --from=build /app/target/*.jar app.jar
RUN addgroup -S appgroup && adduser -S appuser -G appgroup
USER appuser
EXPOSE 8080
ENTRYPOINT ["java", "-jar", "app.jar"]
''')
print("Created project/java-app/{pom.xml, src/main/java/com/example/Main.java, Dockerfile} (multi-stage Maven build)")
print("Build: docker build -t java-app project/java-app")
print("Run:   docker run -p 8080:8080 java-app")


Created project/java-app/Dockerfile (multi-stage Maven build)
Build: docker build -t java-app project/java-app
Run:   docker run -p 8080:8080 java-app


## 5. Dockerfile Best Practices Applied Above
- **Multi-stage builds** (Node/Java) keep the final image small — build tools don't ship to production.
- **Small base images** (`alpine`, `slim`) reduce attack surface and image size.
- **`.dockerignore`** avoids copying `node_modules`/`.git` into the build context.
- **Non-root user** (`USER appuser`) is a security best practice — containers shouldn't run as root by default.
- **Layer caching**: dependency files (`package.json`, `requirements.txt`, `pom.xml`) are copied and installed *before* the rest of the source, so dependency layers are cached across builds when only source code changes.